In [9]:
audio_descriptors = {
    "gender": ["man", "woman"],
    "reverberation": [
        "very distant-sounding", "distant-sounding", "slightly distant-sounding", "slightly close-sounding", "very close-sounding"
    ],
    "noise": [
        "extremely noisy", "very noisy", "noisy", "slightly noisy", "almost no noise", "very clear"
    ],
    "accent": [
        "American", "British", "Chinese", "Indian", "Australian", "Spanish", "French", "German", "Italian", "Russian", "Japanese"
    ],
    "emotion": [
        "angry", "awe", "bored", "calm", "confused", "neutral", "desirous", "disgusted", "enunciated", "fearful", "happy", "laughing", "projected", "sad", "sarcastic", "sleepy", "sympathetic", "whispering"
    ]
}

Returns logically appropriate subsets of tone, pace, and pitch based on the energy/nature of the selected emotion.

In [10]:
import random

def get_constrained_features(emotion):
    low_energy_emotions = ["bored", "calm", "sad", "sleepy", "whispering"]
    high_energy_emotions = ["angry", "fearful", "happy", "laughing", "projected"]

    if emotion in low_energy_emotions:
        # Prevent fast pacing, high pitches, and extreme animation
        tones = ["very monotone", "monotone", "slightly expressive and animated"]
        paces = ["very slowly", "slowly", "slightly slowly", "moderate speed"]
        pitches = ["very low-pitch", "low-pitch", "slightly low-pitch", "moderate pitch"]
    elif emotion in high_energy_emotions:
        # Prevent monotone, slow pacing (angry can still be low pitch, so we keep most pitches)
        tones = ["slightly expressive and animated", "expressive and animated", "very expressive and animated"]
        paces = ["moderate speed", "slightly fast", "fast", "very fast"]
        pitches = ["low-pitch", "slightly low-pitch", "moderate pitch", "slightly high-pitch", "high-pitch", "very high-pitch"]
    else:
        # Flexible/Neutral emotions (awe, confused, disgusted, sarcastic, etc.)
        # Exclude only the absolute extremes to maintain a natural sound
        tones = ["monotone", "slightly expressive and animated", "expressive and animated"]
        paces = ["slowly", "slightly slowly", "moderate speed", "slightly fast", "fast"]
        pitches = ["low-pitch", "slightly low-pitch", "moderate pitch", "slightly high-pitch", "high-pitch"]

    return random.choice(tones), random.choice(paces), random.choice(pitches)

In [12]:
def generate_parler_descriptions(descriptors):
    generated_prompts = []
    for idx in range(3):
        # 1. Select independent traits
        gender = random.choice(descriptors["gender"])
        reverb = random.choice(descriptors["reverberation"])
        noise = random.choice(descriptors["noise"])
        accent = random.choice(descriptors["accent"])
        emotion = random.choice(descriptors["emotion"])

        # 2. Select constrained traits
        tone, pace, pitch = get_constrained_features(emotion)
        pronoun_sub = "She" if gender == "woman" else "He"
        pronoun_obj = "Her" if gender == "woman" else "His"

        # 3. Rule: Randomly omit 'moderate speed' and 'moderate pitch'
        if pace == "moderate speed" and random.choice([True, False]):
            pace_str = ""
        else:
            pace_str = f" at a {pace}" if pace == "moderate speed" else f" {pace}"
        if pitch == "moderate pitch" and random.choice([True, False]):
            pitch_str = ""
        else:
            pitch_str = f" with a {pitch} voice"

        # 4. Rule: Check extreme conditions for recording quality
        quality_prefix = ""
        if noise == "very noisy" and reverb == "very distant-sounding":
            quality_prefix = random.choice(["In a very poor recording, ", "In a very bad recording, "])
        elif noise == "very clear" and reverb == "very close-sounding":
            quality_prefix = random.choice(["In a very good recording, ", "In an excellent recording, "])

        # 5. Define Natural Language Templates integrating the Emotion
        if idx == 0:
            raw_prompt = f"{quality_prefix}A {gender} with a {accent} accent speaks{pace_str} in a {emotion} manner, with a {tone} delivery. The recording is {noise} and {reverb}."
        elif idx == 1:
            raw_prompt = f"{quality_prefix}In a {noise} environment, a {gender} speaker with a {accent} accent delivers a {emotion}, {tone} speech{pace_str}."
        else:
            raw_prompt = f"{quality_prefix}A {gender} with a {accent} accent enunciates a {tone} speech, sounding distinctly {emotion}. {pronoun_obj} voice is {reverb}, with the environment sounding {noise}. {pronoun_sub} speaks{pace_str}{pitch_str}."

        # Clean up any weird spacing left by omitted pace/pitch strings
        clean_prompt = " ".join(raw_prompt.split()).replace(" ,", ",").replace(" .", ".")
        generated_prompts.append(clean_prompt)

    return generated_prompts

Simulating 2 different user commands

In [19]:
for i in range(2):
    print(f"--- Variations for Command {i+1} ---")
    prompts_list = generate_parler_descriptions(audio_descriptors)
    for j, prompt in enumerate(prompts_list):
        print(f"Template {j+1}: {prompt}")

--- Variations for Command 1 ---
Template 1: A woman with a Indian accent speaks very fast in a fearful manner, with a slightly expressive and animated delivery. The recording is slightly noisy and slightly distant-sounding.
Template 2: In a very noisy environment, a woman speaker with a British accent delivers a fearful, slightly expressive and animated speech fast.
Template 3: A woman with a Chinese accent enunciates a monotone speech, sounding distinctly desirous. Her voice is very distant-sounding, with the environment sounding slightly noisy. She speaks slightly slowly with a slightly low-pitch voice.
--- Variations for Command 2 ---
Template 1: A man with a Spanish accent speaks slightly fast in a confused manner, with a monotone delivery. The recording is almost no noise and slightly distant-sounding.
Template 2: In a extremely noisy environment, a woman speaker with a French accent delivers a whispering, slightly expressive and animated speech.
Template 3: A man with a British 